In [40]:
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

tracking_uri = "https://cyrilbrg-mlflow-music.hf.space/"
experiment = "audio_classifier"


def list_mlflow_models(tracking_uri: str) -> list[str]:
    """Récupère la liste des modèles enregistrés dans MLflow."""
    try:
        mlflow.set_tracking_uri(tracking_uri)
        client = mlflow.MlflowClient()
        models = [m.name for m in client.search_registered_models()]
        return models if models else ["(aucun modèle trouvé)"]
    except Exception as e:
        return [f"Erreur MLflow : {e}"]
    
import mlflow
import streamlit as st


def list_mlflow_models2(tracking_uri: str, experiment_name: str = None) -> list[str]:
    """Récupère les modèles enregistrés liés à une expérience spécifique."""
    try:
        mlflow.set_tracking_uri(tracking_uri)
        client = mlflow.MlflowClient()
        
        # Récupérer l'ID de l'expérience si un nom est fourni
        target_exp_id = None
        if experiment_name:
            experiment = client.get_experiment_by_name(experiment_name)
            if not experiment:
                return [f"Erreur : Expérience '{experiment_name}' introuvable."]
            target_exp_id = experiment.experiment_id

        # Récupérer tous les modèles enregistrés
        registered_models = client.search_registered_models()
        
        filtered_models = []
        
        for rm in registered_models:
            if target_exp_id:
                # Garder le modèle si au moins une version vient de l'expérience cible
                if any(client.get_run(v.run_id).info.experiment_id == target_exp_id for v in rm.latest_versions):
                    filtered_models.append(f"{rm.name} (Exp: {experiment_name})")
            else:
                filtered_models.append(rm.name)

        return filtered_models if filtered_models else ["Aucun modèle trouvé"]

    except Exception as e:
        return [f"Erreur MLflow : {str(e)}"]
    
    
liste = list_mlflow_models(tracking_uri)

In [41]:
liste

['BEST_cnn_audio_classifier',
 'MGC_features_SVM_baseline',
 'baseline_cnn_audio_classifier']

In [42]:
from mlflow import MlflowClient

client = MlflowClient()

def get_model_uri(model_name, stage="challenger"):
    return f"models:/{model_name}@{stage}"

# Exemple
model_name = "BEST_cnn_audio_classifier"
model_uri = get_model_uri(model_name)
print(model_uri)

models:/BEST_cnn_audio_classifier@challenger


In [43]:
model_uri = uri

In [44]:
import mlflow
mlflow.pyfunc.get_model_dependencies(model_uri)

RestException: INVALID_PARAMETER_VALUE: Registered model alias challenger not found.

In [45]:
import io


import numpy as np
import pandas as pd
import librosa

TARGET_SR         = 22050
CLIP_DURATION     = 30       # secondes conservées après silence initial
N_FFT             = 2048
HOP               = 512
IMAGE_NX          = 432
IMAGE_NY          = 288 

def preprocess_signal(y, sr) -> tuple:
    """
    Prétraitement pour la prédiction :
      1. Supprime le silence initial
      2. Garde les 3 premières secondes
      3. Rééchantillonne à TARGET_SR
      4. Normalise en amplitude RMS
    """
    y_trimmed, _ = librosa.effects.trim(y)
    n_samples = int(CLIP_DURATION * TARGET_SR)
    y_resampled = librosa.resample(y_trimmed, orig_sr=sr, target_sr=TARGET_SR)
    y_clip = y_resampled[:n_samples] if len(y_resampled) >= n_samples else np.pad(
        y_resampled, (0, n_samples - len(y_resampled))
    )
    # Normalisation RMS
    rms = np.sqrt(np.mean(y_clip ** 2))
    if rms > 0:
        y_clip = y_clip / rms
    return y_clip, TARGET_SR

def compute_features(y, sr) -> np.ndarray:
    """Calcule un vecteur de features (MFCCs, chroma, spectral centroid, etc.).
       TO update"""
    features = []
    column_names = []
    features.append('user.file')
    column_names.append('filename')
    
    length = len(y)
    features.append(length)
    column_names.append('length')
    
    
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    rms   = librosa.feature.rms(y=y)
    spectral_centroid= librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_bandwidth= librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    zero_crossing_rate= librosa.feature.zero_crossing_rate(y)
    harmony, perceptr = librosa.effects.hpss(y)
    tempo, _ = librosa.beat.beat_track(y=y, sr = sr)
    
    mfccs   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    # for x in mfccs:
        
    #     features.append(np.mean(x))
    
    feature_dict = {
    'chroma': chroma,
    'rms': rms,
    'spectral_centroid':spectral_centroid,
    'spectral_bandwidth':spectral_bandwidth,
    'rolloff':rolloff,
    'zero_crossing_rate':zero_crossing_rate,
    'harmony':harmony,
    'perceptr':perceptr,
    'tempo':tempo,
    }
    # add features values and columns_names: 
    for name, data in feature_dict.items():
        features.extend([data.mean(), data.var()])
        column_names.extend([f'{name}_mean', f'{name}_var'])
        
    # add features mfcc1 to mfcc_20 _mean and _var :
    for idx,x in enumerate(mfccs):
        features.extend([np.mean(x),np.var(x)])
        column_names.extend([f"mfcc{idx+1}_mean",f"mfcc{idx+1}_var"])

             
    # add label
    features.append('user')
    column_names.append('label')
    
    
    # columns needed : 
    # """Index(['filename', 'length', 
    # 'chroma_stft_mean', 'chroma_stft_var',
    # 'rms_mean','rms_var', 
    # 'spectral_centroid_mean', 'spectral_centroid_var',
    # 'spectral_bandwidth_mean', 'spectral_bandwidth_var', 
    # 'rolloff_mean','rolloff_var', 
    # 'zero_crossing_rate_mean', 'zero_crossing_rate_var',
    # 'harmony_mean', 'harmony_var', 'perceptr_mean', 'perceptr_var', 
    # 'tempo',
    #    'mfcc1_mean', 'mfcc1_var', 'mfcc2_mean', 'mfcc2_var', 
    #    'mfcc3_mean','mfcc3_var', 'mfcc4_mean', 'mfcc4_var', 
    #    'mfcc5_mean', 'mfcc5_var','mfcc6_mean', 'mfcc6_var', 
    #    'mfcc7_mean', 'mfcc7_var', 'mfcc8_mean',
    #    'mfcc8_var', 'mfcc9_mean', 'mfcc9_var', 'mfcc10_mean', 'mfcc10_var',
    #    'mfcc11_mean', 'mfcc11_var', 'mfcc12_mean', 'mfcc12_var', 'mfcc13_mean',
    #    'mfcc13_var', 'mfcc14_mean', 'mfcc14_var', 'mfcc15_mean', 'mfcc15_var',
    #    'mfcc16_mean', 'mfcc16_var', 'mfcc17_mean', 'mfcc17_var', 'mfcc18_mean',
    #    'mfcc18_var', 'mfcc19_mean', 'mfcc19_var', 'mfcc20_mean', 'mfcc20_var',
    #    'label'],
    # """
    # ONLY 'filename', 'length' at begin, and label at end will be missing
    df_features = pd.DataFrame(columns=column_names)
    df_features.loc[0] = features
    return df_features


def compute_melspectrogram(y, sr) -> np.ndarray:
    """Calcule le mel-spectrogramme normalisé (IMAGE_NY x IMAGE_NX) pour la prédiction."""
    spect = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP)
    spect = librosa.power_to_db(spect, ref=np.max)
    spect.resize(IMAGE_NY, IMAGE_NX, refcheck=False)
    
    return spect


In [47]:
import os
# Cyril data located 2 steps above :
general_path = '../../gtzan-dataset-music-genre-classification/Data'
print(list(os.listdir(f'{general_path}/genres_original/')))
# Importing 1 file
genre = "rock"
audio_file_name = f"{genre}.00053"
y_raw, sr_raw = librosa.load(f"{general_path}/genres_original/{genre}/{audio_file_name}.wav")



model_name_selected= "MGC_features_SVM_baseline"

#y_raw, sr_raw = librosa.load(io.BytesIO(raw_bytes), sr=None)
y_proc, sr_proc = preprocess_signal(y_raw, sr_raw)
feats = compute_features(y_proc, sr_proc)

spect = compute_melspectrogram(y_proc, sr_proc)

list_features_obj = [feats,spect]


features_df = list_features_obj[0]    # is a dataframe
    
image_vector = list_features_obj[1]   # is format of image vector : [[],[],[]] perhaps
image_vect_list = image_vector.tolist()

num_features = features_df.iloc[0].to_dict()
coords = [{"coord":[float(x),float(y)]}
          for x in range(image_vector.shape[0])
          for y in range(image_vector.shape[1])]
coords_list = spect.tolist()  
coords_2 = [{"coord":[0,0]} ]
payload = {
        "model_name": model_name_selected,
        
        "list_features": {
            "num_features": num_features, # [num_features_obj],  # liste d'un seul élément NumFeatures
            "image_coords": coords_2 # [{"coord": coords} for coords in image_coords_list]  # liste d'objets ImageCoord
        }
        
    }


print(payload)

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
{'model_name': 'MGC_features_SVM_baseline', 'list_features': {'num_features': {'filename': 'user.file', 'length': 661500, 'chroma_mean': 0.4091429114341736, 'chroma_var': 0.07498376071453094, 'rms_mean': 0.9744459390640259, 'rms_var': 0.04965870827436447, 'spectral_centroid_mean': 3361.7673451908195, 'spectral_centroid_var': 304378.5059264769, 'spectral_bandwidth_mean': 2933.2024987911896, 'spectral_bandwidth_var': 55104.363716380954, 'rolloff_mean': 6920.5248191998835, 'rolloff_var': 1020520.8547293701, 'zero_crossing_rate_mean': 0.18757331777283281, 'zero_crossing_rate_var': 0.0035673045650667155, 'harmony_mean': 0.0038600314874202013, 'harmony_var': 0.4509669244289398, 'perceptr_mean': 0.009684084914624691, 'perceptr_var': 0.23890294134616852, 'tempo_mean': 117.45383522727273, 'tempo_var': 0.0, 'mfcc1_mean': 164.651611328125, 'mfcc1_var': 1102.639404296875, 'mfcc2_mean': 52.615936279296875

In [ ]:
spect.tolist()

[[-26.886384963989258,
  -30.593420028686523,
  -33.82396697998047,
  -33.97794723510742,
  -37.356956481933594,
  -39.04435348510742,
  -28.866971969604492,
  -16.260879516601562,
  -10.156402587890625,
  -10.708831787109375,
  -15.315799713134766,
  -17.39388084411621,
  -21.402231216430664,
  -27.906574249267578,
  -28.90565299987793,
  -27.161596298217773,
  -29.087360382080078,
  -36.88352584838867,
  -32.265995025634766,
  -29.90726089477539,
  -33.33032989501953,
  -33.414100646972656,
  -32.905113220214844,
  -34.882843017578125,
  -35.21818542480469,
  -37.038848876953125,
  -40.73270034790039,
  -42.392852783203125,
  -35.33277130126953,
  -35.866485595703125,
  -34.989105224609375,
  -33.715782165527344,
  -35.50077819824219,
  -38.263145446777344,
  -32.95211410522461,
  -32.24109649658203,
  -32.47831726074219,
  -33.354454040527344,
  -34.165374755859375,
  -21.707693099975586,
  -12.225311279296875,
  -9.12850570678711,
  -11.118850708007812,
  -14.848600387573242,
  -17

In [ ]:
payload

{'model_name': 'MGC_features_SVM_baseline',
 'list_features': {'num_features': [{'filename': 'user.file',
    'length': 661500,
    'chroma_mean': 0.4091429114341736,
    'chroma_var': 0.07498376071453094,
    'rms_mean': 0.9744459390640259,
    'rms_var': 0.04965870827436447,
    'spectral_centroid_mean': 3361.7673451908195,
    'spectral_centroid_var': 304378.5059264769,
    'spectral_bandwidth_mean': 2933.2024987911896,
    'spectral_bandwidth_var': 55104.363716380954,
    'rolloff_mean': 6920.5248191998835,
    'rolloff_var': 1020520.8547293701,
    'zero_crossing_rate_mean': 0.18757331777283281,
    'zero_crossing_rate_var': 0.0035673045650667155,
    'harmony_mean': 0.0038600314874202013,
    'harmony_var': 0.4509669244289398,
    'perceptr_mean': 0.009684084914624691,
    'perceptr_var': 0.23890294134616852,
    'tempo_mean': 117.45383522727273,
    'tempo_var': 0.0,
    'mfcc1_mean': 164.651611328125,
    'mfcc1_var': 1102.639404296875,
    'mfcc2_mean': 52.615936279296875,
   

In [ ]:
target_json = "{
  "model_name": "string",
  "list_features": {
    "num_features": [
      {
        "filename": "string",
        "length": 0,
        "chroma_mean": 0,
        "chroma_var": 0,
        "rms_mean": 0,
        "rms_var": 0,
        "spectral_centroid_mean": 0,
        "spectral_centroid_var": 0,
        "spectral_bandwidth_mean": 0,
        "spectral_bandwidth_var": 0,
        "rolloff_mean": 0,
        "rolloff_var": 0,
        "zero_crossing_rate_mean": 0,
        "zero_crossing_rate_var": 0,
        "harmony_mean": 0,
        "harmony_var": 0,
        "perceptr_mean": 0,
        "perceptr_var": 0,
        "tempo": 0,
        "mfcc1_mean": 0,
        "mfcc1_var": 0,
        "mfcc2_mean": 0,
        "mfcc2_var": 0,
        "mfcc3_mean": 0,
        "mfcc3_var": 0,
        "mfcc4_mean": 0,
        "mfcc4_var": 0,
        "mfcc5_mean": 0,
        "mfcc5_var": 0,
        "mfcc6_mean": 0,
        "mfcc6_var": 0,
        "mfcc7_mean": 0,
        "mfcc7_var": 0,
        "mfcc8_mean": 0,
        "mfcc8_var": 0,
        "mfcc9_mean": 0,
        "mfcc9_var": 0,
        "mfcc10_mean": 0,
        "mfcc10_var": 0,
        "mfcc11_mean": 0,
        "mfcc11_var": 0,
        "mfcc12_mean": 0,
        "mfcc12_var": 0,
        "mfcc13_mean": 0,
        "mfcc13_var": 0,
        "mfcc14_mean": 0,
        "mfcc14_var": 0,
        "mfcc15_mean": 0,
        "mfcc15_var": 0,
        "mfcc16_mean": 0,
        "mfcc16_var": 0,
        "mfcc17_mean": 0,
        "mfcc17_var": 0,
        "mfcc18_mean": 0,
        "mfcc18_var": 0,
        "mfcc19_mean": 0,
        "mfcc19_var": 0,
        "mfcc20_mean": 0,
        "mfcc20_var": 0,
        "label": 0
      }
    ],
    "image_coords": [
      {
        "coord": [
          0,
          0
        ]
      }
    ]
  }
}
"

SyntaxError: unterminated string literal (detected at line 1) (2488410066.py, line 1)